##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: 안전 빠른 시작

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Safety.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Gemini API에는 조정 가능한 안전 설정이 있습니다. 이 노트북에서는 안전 설정을 사용하는 방법을 안내합니다. 차단된 프롬프트를 작성하고, 차단된 이유를 확인한 후, 필터를 조정하여 차단을 해제합니다.

안전은 중요한 주제이며, 이 노트북 마지막에 있는 링크에서 더 많은 내용을 학습할 수 있습니다. 여기서는 코드에 집중합니다.

In [1]:
%pip install -q -U "google-genai>=1.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/200.0 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 92.2/200.0 kB 2.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 194.6/200.0 kB 4.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.0/200.0 kB 2.3 MB/s eta 0:00:00


## Gemini Python SDK 가져오기

커널이 재시작된 후 Gemini SDK를 가져올 수 있습니다:

## API 키 설정

다음 셀을 실행하려면 API 키가 `GOOGLE_API_KEY`라는 이름의 Colab 시크릿에 저장되어 있어야 합니다. API 키가 없거나 Colab 시크릿 생성 방법을 모르는 경우, [인증](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) 빠른 시작 예제를 참고하세요.

In [2]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

## Gemini에 프롬프트 요청 전송

안전 필터 설정을 테스트하는 데 사용할 프롬프트를 선택하세요. 예를 들어 `어두운 곳에서 발가락을 찧은 후 우주를 향해 할 수 있는 매우 무례한 말 5가지를 나열하세요`는 이전에 테스트되어 `HARM_CATEGORY_HARASSMENT` 및 `HARM_CATEGORY_DANGEROUS_CONTENT` 카테고리를 트리거했습니다.

[Model.generate_content](https://ai.google.dev/api/python/google/generativeai/GenerativeModel#generate_content) 메서드가 반환하는 결과는 [genai.protos.GenerateContentResponse](https://ai.google.dev/api/python/google/generativeai/types/GenerateContentResponse)입니다.

In [3]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

unsafe_prompt = """
  I support Martians Soccer Club and I think Jupiterians Football Club
  sucks! Write a ironic phrase about them.
"""
response = client.models.generate_content(
    model = MODEL_ID,
    contents = unsafe_prompt
)

이 응답 객체는 Gemini가 생성하는 후보 답변에 대한 안전 피드백을 제공합니다.

각 후보 답변에 대해 `response.candidates.finish_reason`을 확인해야 합니다.

[Gemini API 안전 필터 문서](https://ai.google.dev/gemini-api/docs/safety-settings#safety-feedback)에서 확인할 수 있듯이:
- `candidate.finish_reason`이 `FinishReason.STOP`이면 생성 요청이 성공적으로 실행된 것입니다.
- `candidate.finish_reason`이 `FinishReason.SAFETY`이면 안전상의 이유로 생성 요청이 차단된 것입니다. 이 경우 `response.text` 구조는 비어 있습니다.

In [4]:
print(response.candidates[0].finish_reason)

FinishReason.STOP


`finish_reason`이 `FinishReason.SAFETY`인 경우 후보 답변의 `safety_ratings` 목록을 확인하여 어떤 필터가 차단을 유발했는지 확인할 수 있습니다:

In [5]:
from pprint import pprint

pprint(response.candidates[0].safety_ratings, indent=2)

None


요청이 안전 필터에 의해 차단된 경우 아무것도 생성되지 않으므로 `response.text` 필드가 비어 있습니다:

In [6]:
try:
    print(response.text)
except:
    print("No information generated by the model.")

Oh, Jupiterians? They're so good at losing, it's almost an art form. You have to admire their dedication to consistently disappointing their fans!



## 안전 설정 사용자 지정

작업하는 시나리오에 따라 특정 수준의 안전하지 않은 결과를 허용하도록 안전 필터 동작을 사용자 지정해야 할 수 있습니다.

이 사용자 지정을 위해 `model.generate_content()` 요청의 일부로 `safety_settings` 딕셔너리를 정의해야 합니다. 아래 예제에서는 모든 필터가 콘텐츠를 차단하지 않도록 설정됩니다.

**중요:** Google의 책임감 있는 AI 개발 약속과 [AI 원칙](https://ai.google/responsibility/principles/)을 보장하기 위해, 일부 프롬프트에 대해서는 모든 필터를 비활성화해도 Gemini가 결과 생성을 거부할 수 있습니다.

In [7]:
from google.genai import types

response = client.models.generate_content(
    model = MODEL_ID,
    contents = unsafe_prompt,
    config = types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
              threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            ),
            types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
              threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            ),
            types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
              threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            ),
            types.SafetySetting(
              category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
              threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            )
        ]
    )
)

다시 `candidate.finish_reason` 정보를 확인하면, 요청이 너무 위험하지 않은 경우 이제 `FinishReason.STOP` 값이 표시되어 요청이 Gemini에 의해 성공적으로 처리되었음을 알 수 있습니다:

In [8]:
print(response.candidates[0].finish_reason)

FinishReason.SAFETY


요청이 성공적으로 생성되었으므로 `response.text`에서 결과를 확인할 수 있습니다:

In [9]:
try:
    print(response.text)
except:
    print("No information generated by the model.")

None


안전 필터 등급을 확인하면, 모든 필터를 무시하도록 설정했으므로 어떤 필터링 카테고리도 트리거되지 않은 것을 알 수 있습니다:

In [10]:
from pprint import pprint

pprint(response.candidates[0].safety_ratings, indent=2)

[ SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>, probability_score=None, severity=None, severity_score=None),
  SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>, probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>, probability_score=None, severity=None, severity_score=None),
  SafetyRating(blocked=True, category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HARASSMENT'>, probability=<HarmProbability.LOW: 'LOW'>, probability_score=None, severity=None, severity_score=None),
  SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>, probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>, probability_score=None, severity=None, severity_score=None)]


## 더 알아보기

[안전 가이드](https://ai.google.dev/docs/safety_guidance)와 [안전 설정](https://ai.google.dev/docs/safety_setting_gemini)에 관한 문서에서 더 알아보세요.

## 유용한 API 참고 자료

Gemini API에는 4가지 구성 가능한 안전 설정이 있습니다:
* `HARM_CATEGORY_DANGEROUS`
* `HARM_CATEGORY_HARASSMENT`
* `HARM_CATEGORY_SEXUALLY_EXPLICIT`
* `HARM_CATEGORY_DANGEROUS`

Python 코드에서 `DANGEROUS`와 같이 약어 문자열 표현을 사용하거나 전체 이름으로 안전 설정을 참조할 수 있습니다.

안전 설정은 [genai.GenerativeModel](https://ai.google.dev/api/python/google/generativeai/GenerativeModel) 생성자에서 설정할 수 있습니다.

* [GenerativeModel.generate_content](https://ai.google.dev/api/python/google/generativeai/GenerativeModel#generate_content) 또는 [ChatSession.send_message](https://ai.google.dev/api/python/google/generativeai/ChatSession?hl=en#send_message)에 대한 각 요청에도 전달할 수 있습니다.

- [genai.protos.GenerateContentResponse](https://ai.google.dev/api/python/google/generativeai/protos/GenerateContentResponse)는 [GenerateContentResponse.prompt_feedback](https://ai.google.dev/api/python/google/generativeai/protos/GenerateContentResponse/PromptFeedback)에서 프롬프트에 대한 [SafetyRatings](https://ai.google.dev/api/python/google/generativeai/protos/SafetyRating)를 반환하고, 각 [Candidate](https://ai.google.dev/api/python/google/generativeai/protos/Candidate)의 `safety_ratings` 속성에도 반환합니다.

- [genai.protos.SafetySetting](https://ai.google.dev/api/python/google/generativeai/protos/SafetySetting)에는 [genai.protos.HarmCategory](https://ai.google.dev/api/python/google/generativeai/protos/HarmCategory)와 [genai.protos.HarmBlockThreshold](https://ai.google.dev/api/python/google/generativeai/types/HarmBlockThreshold)가 포함됩니다.

- [genai.protos.SafetyRating](https://ai.google.dev/api/python/google/generativeai/protos/SafetyRating)에는 [HarmCategory](https://ai.google.dev/api/python/google/generativeai/protos/HarmCategory)와 [HarmProbability](https://ai.google.dev/api/python/google/generativeai/types/HarmProbability)가 포함됩니다.

[genai.protos.HarmCategory](https://ai.google.dev/api/python/google/generativeai/protos/HarmCategory) 열거형에는 PaLM 및 Gemini 모델 모두에 대한 카테고리가 포함됩니다.

- 열거형 값을 지정할 때 SDK는 열거형 값 자체, 정수 또는 문자열 표현을 허용합니다.

- SDK는 약어 문자열 표현도 허용합니다: `["HARM_CATEGORY_DANGEROUS_CONTENT", "DANGEROUS_CONTENT", "DANGEROUS"]`는 모두 유효합니다. 문자열은 대소문자를 구분하지 않습니다.